# 03 — Model Training: EfficientNet-B3 for Diabetic Retinopathy Grading

Trains the grading model on the canonical preprocessed dataset from `02_preprocessing` and saves the
best checkpoint for evaluation.

**Model.** EfficientNet-B3, ImageNet-pretrained, two-stage transfer learning: (1) train the new
5-class head with the backbone frozen, then (2) unfreeze and fine-tune the whole network at a much
lower learning rate. The best checkpoint is selected on **validation QWK**, the ordinal metric that
matches DR severity, rather than accuracy.

**Consumes** (from `02_preprocessing`)
- `artifacts/preprocessing/aptos_preprocessed_384_manifest.csv` — id, label, split, image path.
- `artifacts/preprocessing/class_weights_effective_number.npy` — training-split class weights.

**Produces**
- `artifacts/checkpoints/efficientnet_b3_best_qwk.pt` — best-QWK checkpoint (+ metadata).
- `artifacts/results/efficientnet_b3_training_history.csv` — per-epoch metrics.

## 0. Setup

In [1]:
# Version record for reproducibility.
import sys, platform
import numpy as np, pandas as pd, torch, torchvision, sklearn

print("python      ", platform.python_version())
for m in (np, pd, torch, torchvision, sklearn):
    print(f"{m.__name__:<12}", m.__version__)

python       3.11.15
numpy        2.4.4
pandas       3.0.3
torch        2.13.0+cu126
torchvision  0.28.0+cu126
sklearn      1.9.0


In [2]:
import copy, random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import EfficientNet_B3_Weights
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             balanced_accuracy_score, accuracy_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# benchmark=True favours speed; it does not guarantee bitwise determinism.
torch.backends.cudnn.benchmark = True

ID_COL, LABEL_COL = "id_code", "diagnosis"
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]

# --- Training config ---
IMG_SIZE        = 384          # must match the preprocessed images
BATCH_SIZE      = 8
NUM_WORKERS     = 0
HEAD_EPOCHS     = 5
FINETUNE_EPOCHS = 25
PATIENCE        = 7

# --- Paths (contract with 02_preprocessing) ---
MANIFEST_PATH = Path(f"artifacts/preprocessing/aptos_preprocessed_{IMG_SIZE}_manifest.csv")
WEIGHTS_PATH  = Path("artifacts/preprocessing/class_weights_effective_number.npy")
CHECKPOINT_DIR = Path("artifacts/checkpoints")
RESULTS_DIR    = Path("artifacts/results")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "efficientnet_b3_best_qwk.pt"
HISTORY_PATH    = RESULTS_DIR / "efficientnet_b3_training_history.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
assert MANIFEST_PATH.exists(), f"{MANIFEST_PATH} not found — run 02_preprocessing.ipynb first"

Device: cuda


c:\Users\ritik\anaconda3\envs\cv-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the preprocessing manifest

Reuses the exact train/val/test split from EDA and preprocessing — no re-splitting here.

In [3]:
manifest = pd.read_csv(MANIFEST_PATH)

required = {ID_COL, LABEL_COL, "split", "preprocessed_path"}
missing = required - set(manifest.columns)
assert not missing, f"Manifest missing columns: {sorted(missing)}"
assert set(manifest["split"]) == {"train", "val", "test"}, "Unexpected split labels."

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df   = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_df  = manifest[manifest["split"] == "test"].reset_index(drop=True)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

print(train_df[LABEL_COL].value_counts().sort_index()
      .rename(index={i: CLASS_NAMES[i] for i in range(5)}).to_string())

train 2563 | val 549 | test 550
diagnosis
No DR            1263
Mild              259
Moderate          699
Severe            135
Proliferative     207


## 2. Transforms, dataset, loaders

The saved images are already cropped, CLAHE-enhanced and resized to 384, so no resizing here.
Augmentation is applied to the training split only; ImageNet normalisation is applied to all splits
at load time. The dataset returns the image id as well, for downstream error analysis and XAI.

In [4]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomAffine(degrees=15, translate=(0.04, 0.04), scale=(0.95, 1.05),
                            interpolation=transforms.InterpolationMode.BILINEAR, fill=0),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05, hue=0.01),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class APTOSDataset(Dataset):
    def __init__(self, frame, transform):
        self.df, self.transform = frame.reset_index(drop=True), transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["preprocessed_path"]).convert("RGB")
        return self.transform(img), int(row[LABEL_COL]), row[ID_COL]

train_dataset = APTOSDataset(train_df, train_transform)
val_dataset   = APTOSDataset(val_df, eval_transform)
test_dataset  = APTOSDataset(test_df, eval_transform)

# Guard: preprocessed images must be IMG_SIZE, since no resize happens here.
_img, _, _ = val_dataset[0]
assert tuple(_img.shape) == (3, IMG_SIZE, IMG_SIZE), f"Unexpected image shape {_img.shape}"

def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

train_loader = make_loader(train_dataset, True)
val_loader   = make_loader(val_dataset, False)
test_loader  = make_loader(test_dataset, False)
print("DataLoaders ready.")

DataLoaders ready.


## 3. Class-weighted loss

Class weights come from the training split only (effective-number scheme from preprocessing), so
minority-grade errors count more without permanently rebalancing the data. Mild label smoothing
(0.05) regularises the confident majority class. If the weight artifact is missing, it is recomputed
with the same formula as a fallback.

In [5]:
if WEIGHTS_PATH.exists():
    class_weights = np.load(WEIGHTS_PATH).astype(np.float32)
else:
    counts = (train_df[LABEL_COL].value_counts().sort_index()
              .reindex(range(5), fill_value=0).values.astype(np.float64))
    beta = 0.999
    w = (1.0 - beta) / (1.0 - np.power(beta, counts))
    w = w / w.mean()
    w = np.minimum(w, 3.0)
    class_weights = (w / w.mean()).astype(np.float32)

loss_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=0.05)
print("Class weights:", np.round(class_weights, 4))

Class weights: [0.3315 1.0419 0.4727 1.8824 1.2714]


## 4. Build EfficientNet-B3

The ImageNet classifier is replaced with a dropout + 5-class linear head.

In [6]:
model = models.efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.35, inplace=True),
    nn.Linear(in_features, 5),
)
model = model.to(device)
print(model.classifier)

Sequential(
  (0): Dropout(p=0.35, inplace=True)
  (1): Linear(in_features=1536, out_features=5, bias=True)
)


## 5. Metrics and evaluation

QWK (quadratic-weighted) is the primary metric; accuracy, balanced accuracy and macro-F1 are tracked
alongside it.

In [7]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy":          accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1":          f1_score(y_true, y_pred, average="macro", zero_division=0),
        "qwk":               cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }

@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    """Loss + metrics over a loader (no gradient)."""
    model.eval()
    running_loss, y_true, y_pred = 0.0, [], []
    for images, labels, _ in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        logits = model(images)
        running_loss += criterion(logits, labels).item() * images.size(0)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(logits.argmax(1).cpu().tolist())
    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)
    return metrics

## 6. Training loop with early stopping

Each epoch logs train/val metrics; the checkpoint is (over)written whenever validation QWK improves.
Training stops after `PATIENCE` epochs without improvement, and the best weights are restored.

In [8]:
def train_stage(model, train_loader, val_loader, criterion, optimizer, scheduler,
                epochs, stage_name, best_qwk=-np.inf, patience=PATIENCE):
    """Run one training stage; persist the best-QWK checkpoint; return (model, history, best_qwk)."""
    history, best_state, stale = [], None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, y_true, y_pred = 0.0, [], []
        bar = tqdm(train_loader, desc=f"{stage_name} | epoch {epoch}/{epochs}", leave=False)
        for images, labels, _ in bar:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            y_true.extend(labels.detach().cpu().tolist())
            y_pred.extend(logits.argmax(1).detach().cpu().tolist())
            bar.set_postfix(loss=f"{loss.item():.4f}")

        train_metrics = compute_metrics(y_true, y_pred)
        train_metrics["loss"] = running_loss / len(train_loader.dataset)
        val_metrics = evaluate_loader(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()          # cosine schedule steps once per epoch

        history.append({"stage": stage_name, "epoch": epoch,
                        **{f"train_{k}": v for k, v in train_metrics.items()},
                        **{f"val_{k}": v for k, v in val_metrics.items()},
                        "lr": optimizer.param_groups[0]["lr"]})
        print(f"{stage_name} {epoch:02d} | train loss={train_metrics['loss']:.4f} "
              f"| val loss={val_metrics['loss']:.4f} | val QWK={val_metrics['qwk']:.4f} "
              f"| val macro-F1={val_metrics['macro_f1']:.4f} | val bal-acc={val_metrics['balanced_accuracy']:.4f}")

        if val_metrics["qwk"] > best_qwk:
            best_qwk, best_state, stale = val_metrics["qwk"], copy.deepcopy(model.state_dict()), 0
            torch.save({"model_state_dict": best_state, "best_val_qwk": best_qwk,
                        "class_names": CLASS_NAMES, "img_size": IMG_SIZE,
                        "model_name": "efficientnet_b3", "class_weights": class_weights.tolist()},
                       BEST_MODEL_PATH)
            print(f"  new best QWK={best_qwk:.4f} -> {BEST_MODEL_PATH.name}")
        else:
            stale += 1
            if stale >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_qwk

## 7. Stage 1 — train the classifier head

Backbone frozen; only the new head learns, at a relatively high learning rate.

In [9]:
for p in model.features.parameters():   p.requires_grad = False
for p in model.classifier.parameters(): p.requires_grad = True

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, HEAD_EPOCHS))

model, history_head, best_qwk = train_stage(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    epochs=HEAD_EPOCHS, stage_name="head")

head 01 | train loss=1.3681 | val loss=1.2179 | val QWK=0.7563 | val macro-F1=0.4915 | val bal-acc=0.5320
  new best QWK=0.7563 -> efficientnet_b3_best_qwk.pt


head 02 | train loss=1.2258 | val loss=1.1799 | val QWK=0.7636 | val macro-F1=0.5729 | val bal-acc=0.5947
  new best QWK=0.7636 -> efficientnet_b3_best_qwk.pt


head 03 | train loss=1.1746 | val loss=1.1639 | val QWK=0.7677 | val macro-F1=0.5353 | val bal-acc=0.5308
  new best QWK=0.7677 -> efficientnet_b3_best_qwk.pt


head 04 | train loss=1.1588 | val loss=1.1853 | val QWK=0.7758 | val macro-F1=0.5258 | val bal-acc=0.5753
  new best QWK=0.7758 -> efficientnet_b3_best_qwk.pt


head 05 | train loss=1.1475 | val loss=1.2140 | val QWK=0.7436 | val macro-F1=0.5509 | val bal-acc=0.5601


## 8. Stage 2 — fine-tune the full network

Unfreeze everything and continue at a much smaller learning rate. `best_qwk` carries over, so the
checkpoint is only replaced if fine-tuning genuinely improves on stage 1.

In [10]:
for p in model.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, FINETUNE_EPOCHS), eta_min=1e-6)

model, history_ft, best_qwk = train_stage(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    epochs=FINETUNE_EPOCHS, stage_name="finetune", best_qwk=best_qwk)

finetune 01 | train loss=1.1110 | val loss=1.1218 | val QWK=0.7821 | val macro-F1=0.5730 | val bal-acc=0.5814
  new best QWK=0.7821 -> efficientnet_b3_best_qwk.pt


finetune 02 | train loss=1.0495 | val loss=1.0958 | val QWK=0.8061 | val macro-F1=0.5939 | val bal-acc=0.6200
  new best QWK=0.8061 -> efficientnet_b3_best_qwk.pt


finetune 03 | train loss=0.9959 | val loss=1.0902 | val QWK=0.8166 | val macro-F1=0.6186 | val bal-acc=0.6194
  new best QWK=0.8166 -> efficientnet_b3_best_qwk.pt


finetune 04 | train loss=0.9621 | val loss=1.0755 | val QWK=0.8200 | val macro-F1=0.6226 | val bal-acc=0.6338
  new best QWK=0.8200 -> efficientnet_b3_best_qwk.pt


finetune 05 | train loss=0.9491 | val loss=1.0366 | val QWK=0.8314 | val macro-F1=0.6355 | val bal-acc=0.6560
  new best QWK=0.8314 -> efficientnet_b3_best_qwk.pt


...

finetune 21 | train loss=0.6988 | val loss=1.0607 | val QWK=0.8551 | val macro-F1=0.6437 | val bal-acc=0.6499


finetune 22 | train loss=0.6848 | val loss=1.1155 | val QWK=0.8436 | val macro-F1=0.6275 | val bal-acc=0.6283


finetune 23 | train loss=0.7004 | val loss=1.0903 | val QWK=0.8361 | val macro-F1=0.6286 | val bal-acc=0.6446


finetune 24 | train loss=0.6915 | val loss=1.0696 | val QWK=0.8560 | val macro-F1=0.6465 | val bal-acc=0.6582


finetune 25 | train loss=0.6726 | val loss=1.0519 | val QWK=0.8556 | val macro-F1=0.6561 | val bal-acc=0.6715


## 9. Save history and plot curves

In [11]:
history = pd.concat([history_head, history_ft], ignore_index=True)
history.to_csv(HISTORY_PATH, index=False)
print("Best validation QWK:", round(best_qwk, 4))
print("Checkpoint:", BEST_MODEL_PATH)
history.tail()

Best validation QWK: 0.8579
Checkpoint: artifacts\checkpoints\efficientnet_b3_best_qwk.pt


,stage,epoch,train_accuracy,train_balanced_accuracy,train_macro_f1,train_qwk,train_loss,val_accuracy,val_balanced_accuracy,val_macro_f1,val_qwk,val_loss,lr
25,finetune,21,0.900507,0.834982,0.832336,0.935002,0.698795,0.785064,0.649919,0.643726,0.855053,1.060669,0.000002
26,finetune,22,0.905579,0.851202,0.844980,0.944478,0.684799,0.774135,0.628331,0.627452,0.843581,1.115462,0.000002
27,finetune,23,0.893484,0.843620,0.830758,0.927308,0.700433,0.765027,0.644640,0.628624,0.836108,1.090296,0.000001
28,finetune,24,0.891533,0.842166,0.828295,0.933698,0.691478,0.783242,0.658159,0.646537,0.856019,1.069553,0.000001
29,finetune,25,0.906360,0.863035,0.848429,0.941437,0.672625,0.788707,0.671530,0.656074,0.855560,1.051886,0.000001


In [ ]:
if len(history):
    x = np.arange(1, len(history) + 1)
    boundary = len(history_head) + 0.5   # head → fine-tune transition

    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(x, history["train_loss"], label="train")
    ax[0].plot(x, history["val_loss"], label="val")
    ax[0].axvline(boundary, color="gray", ls="--", lw=1)
    ax[0].set(title="Loss", xlabel="epoch", ylabel="loss"); ax[0].legend(); ax[0].grid(alpha=0.25)

    for col, lab in [("val_qwk", "QWK"), ("val_macro_f1", "macro-F1"),
                     ("val_balanced_accuracy", "balanced acc")]:
        ax[1].plot(x, history[col], label=lab)
    ax[1].axvline(boundary, color="gray", ls="--", lw=1, label="head → fine-tune")
    ax[1].set(title="Validation metrics", xlabel="epoch", ylabel="score"); ax[1].legend(); ax[1].grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "efficientnet_b3_training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

## Handoff to evaluation

Run `04_evaluation.ipynb` next. It reloads the best-QWK checkpoint
(`artifacts/checkpoints/efficientnet_b3_best_qwk.pt`), evaluates the untouched **test** split from the
same manifest exactly once, and produces the confusion matrix, per-class metrics and predictions.

For Grad-CAM / attribution in the evaluation stage, the EfficientNet-B3 target layer is
`model.features[-1]` (the final convolutional block) — not `layer4`, which belongs to ResNet.

## Appendix — requirements.txt

Same pinned file as the earlier stages (repo root):

```
numpy==1.26.4
pandas==2.2.2
scikit-learn==1.4.2
matplotlib==3.8.4
tqdm==4.66.4
torch==2.2.2
torchvision==0.17.2
pillow==10.3.0
```